# 03 - Data Structures and Algorithms (Python)

This notebook builds the fault-stack / action-queue / CAN-ID-lookup example from `concept.md` idiomatically in Python, then runs a small Big-O demo comparing linear search against a dict lookup. Read `concept.md` first if you haven't.

## Stack: Recent Controller Faults

Python's built-in `list` already supports stack operations directly: `.append()` pushes onto the end, `.pop()` removes and returns from the end. We treat "the end of the list" as "the top of the stack."

In [1]:
fault_stack = []

fault_stack.append("CAN timeout: device 12")
fault_stack.append("Brownout detected")
fault_stack.append("CAN timeout: device 7")

print("Most recent fault first:")
while fault_stack:
    print(" -", fault_stack.pop())

Most recent fault first:
 - CAN timeout: device 7
 - Brownout detected
 - CAN timeout: device 12


Notice the last fault pushed ("CAN timeout: device 7") is the first one popped — Last In, First Out. This is exactly what you want for a driver-station fault display: whatever just went wrong is the first thing shown.

## Queue: Autonomous Action Sequence

A plain `list` *can* act as a queue with `.pop(0)`, but that's O(n) every time (removing from the front means shifting every remaining element over). `collections.deque` is built specifically to make both ends fast — `.append()` to enqueue, `.popleft()` to dequeue.

In [2]:
from collections import deque

action_queue = deque()
action_queue.append("drive forward")
action_queue.append("intake")
action_queue.append("shoot")
action_queue.append("drive back")

print("Executing in queued order:")
while action_queue:
    print(" -", action_queue.popleft())

Executing in queued order:
 - drive forward
 - intake
 - shoot
 - drive back


Here the *first* action queued ("drive forward") is the *first* one executed — First In, First Out. That's the whole point: autonomous steps have to run in the order they were planned, not reversed.

## Hashmap: CAN ID to Device Name

Python's `dict` is a hashmap. Looking up a value by key doesn't require scanning every entry — Python computes a hash of the key and goes almost straight to the matching slot.

In [3]:
can_id_to_name = {
    1: "Front Left Drive",
    2: "Front Right Drive",
    3: "Back Left Drive",
    4: "Back Right Drive",
    12: "Intake Roller",
}

print(can_id_to_name[12])
print(can_id_to_name.get(99, "<unknown device>"))  # .get() avoids a crash on a missing key

Intake Roller
<unknown device>


`can_id_to_name[12]` goes directly to "Intake Roller" regardless of how many other devices are in the map. `.get(key, default)` is the safe way to look up a key that might not exist, without an extra `if key in can_id_to_name` check first.

## Big-O in Practice: Linear Search vs. Dict Lookup

To make the O(n) vs. O(1) difference concrete instead of abstract, we'll build a much bigger table of (id, name) pairs and count how many comparisons a linear scan needs to find a given ID, at three different positions: near the front, near the back, and missing entirely.

In [4]:
def linear_search_comparisons(entries, target_id):
    """Scan a list of (id, name) tuples from the front. Return the number
    of comparisons made before finding target_id (or exhausting the list)."""
    comparisons = 0
    for entry_id, _name in entries:
        comparisons += 1
        if entry_id == target_id:
            return comparisons
    return comparisons


# A big table: 10,000 fake CAN IDs, in order.
big_table = [(i, f"device-{i}") for i in range(10_000)]
big_map = dict(big_table)

for target in [5, 5_000, 9_999]:
    comparisons = linear_search_comparisons(big_table, target)
    print(f"linear search for id={target:>5}: {comparisons:>5} comparisons")

print()
print(f"dict lookup for id=5:     {big_map[5]}")
print(f"dict lookup for id=9999:  {big_map[9999]}")
print("(a dict lookup does roughly the same amount of work no matter where the entry is)")

linear search for id=    5:     6 comparisons
linear search for id= 5000:  5001 comparisons
linear search for id= 9999: 10000 comparisons

dict lookup for id=5:     device-5
dict lookup for id=9999:  device-9999
(a dict lookup does roughly the same amount of work no matter where the entry is)


The comparison count for linear search grows with the target's position — looking up ID `9999` takes about 2,000x more comparisons than looking up ID `5` in this table. The dict lookups above do essentially the same amount of work either way. That growing-with-position behavior *is* what O(n) means in practice; the flat behavior *is* what O(1) means in practice.

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor.

1. Add a `peek()`-style check to the fault stack example that looks at the most recent fault *without* removing it (hint: list indexing).
2. Modify `linear_search_comparisons` to also work for a target that isn't in the table at all, and confirm it returns the full length of `entries`.
3. Time (with `import time; time.perf_counter()`) an actual dict lookup versus an actual `linear_search_comparisons` call for `target = 9_999` on `big_table`/`big_map`, over many repetitions, and see if the wall-clock gap matches what the comparison counts predicted.

In [5]:
# Your code here


## Resources

- [Python `collections.deque` docs](https://docs.python.org/3/library/collections.html#collections.deque) - the real implementation behind the queue example above.
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/) - time/space complexity for common data structures and algorithms.
- [WPILib `SequentialCommandGroup`](https://docs.wpilib.org/en/stable/docs/software/commandbased/command-groups.html) - the real queue-like structure behind chained autonomous actions on the robot.